# Aegis Lens — Python SDK Quick-start

This notebook walks through the key features of the Aegis Lens Python SDK:

1. **Install & import** the SDK
2. **Authenticate** and list events
3. **Analyse** events with pandas
4. **Visualise** event locations on a map

Set `AEGIS_API_KEY` in your environment or replace the placeholder below.

In [ ]:
# Cell 1 — Install + imports
# Run once to install the SDK and its dependencies
%pip install aegis-sdk pandas folium --quiet

import os
import json
import pandas as pd

from aegis import AegisClient

# Set your API key here or via AEGIS_API_KEY environment variable
API_KEY = os.environ.get('AEGIS_API_KEY', 'ak_your_key_here')
BASE_URL = os.environ.get('AEGIS_BASE_URL', 'https://aegislens.io')

client = AegisClient(api_key=API_KEY, base_url=BASE_URL)
print('Client initialised:', client)

In [ ]:
# Cell 2 — Authenticate + list recent events
page = client.events.list(country='UA', hours=24, limit=50)
print(f'Events in last 24h: {page.meta.total} total, fetched {len(page.data)}')

# Flatten events to a list of dicts for pandas
rows = []
for e in page.data:
    rows.append({
        'event_id':    e.event_id,
        'occurred_at': e.occurred_at,
        'class':       e.event_class,
        'severity':    e.severity,
        'danger_score': e.danger_score,
        'confidence':  e.confidence,
        'lat':         e.location.lat if e.location else None,
        'lon':         e.location.lon if e.location else None,
        'summary_en':  (e.summary or {}).get('en', ''),
        'verification': e.verification_state,
    })

df = pd.DataFrame(rows)
df['occurred_at'] = pd.to_datetime(df['occurred_at'])
df.head()

In [ ]:
# Cell 3 — Basic analysis with pandas
print('=== Events by class ===')
print(df['class'].value_counts().to_string())

print('\n=== Average danger score by class ===')
print(df.groupby('class')['danger_score'].mean().sort_values(ascending=False).to_string())

print('\n=== Severity distribution ===')
print(df['severity'].value_counts().sort_index().to_string())

print('\n=== Verification states ===')
print(df['verification'].value_counts().to_string())

# Top 5 highest-danger events
print('\n=== Top 5 by danger score ===')
top5 = df.nlargest(5, 'danger_score')[['class', 'severity', 'danger_score', 'summary_en']]
print(top5.to_string(index=False))

In [ ]:
# Cell 4 — Map visualisation (requires folium)
try:
    import folium

    # Centre on Ukraine
    m = folium.Map(location=[49.0, 32.0], zoom_start=6, tiles='CartoDB dark_matter')

    SEVERITY_COLOUR = {1: 'blue', 2: 'orange', 3: 'red'}

    geo_df = df.dropna(subset=['lat', 'lon'])
    for _, row in geo_df.iterrows():
        colour = SEVERITY_COLOUR.get(int(row['severity']), 'gray')
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=5 + row['danger_score'] * 2,
            color=colour,
            fill=True,
            fill_opacity=0.7,
            popup=folium.Popup(
                f"<b>{row['class']}</b><br>{row['summary_en'][:120]}",
                max_width=300,
            ),
            tooltip=f"{row['class']} — danger {row['danger_score']:.2f}",
        ).add_to(m)

    print(f'Plotted {len(geo_df)} events with coordinates.')
    display(m)  # type: ignore[name-defined]  # noqa: F821

except ImportError:
    print('folium not installed. Run: pip install folium')
    print('Event coordinates sample:')
    print(df[['lat', 'lon', 'class', 'danger_score']].dropna().head(10).to_string())